# Justiça do Trabalho

Algoritmo: HDBSCAN
Parâmetro: 294.00

In [ ]:
%pip install -U pm4py tqdm

import pm4py
from google.colab import drive
import os
import sys

In [ ]:
if not os.path.exists('/content/drive'):
    print("Montando Google Drive...")
    drive.mount('/content/drive')
    print("Google Drive montado.")
else:
    print("Google Drive já está montado.")

EXPERIMENT_PATH = '/content/drive/MyDrive/experimentos_mp'
UTIL_PATH = f'{EXPERIMENT_PATH}/utils'
if UTIL_PATH not in sys.path:
    sys.path.append(UTIL_PATH)
    print(f"'{UTIL_PATH}' adicionado ao sys.path.")
else:
    print(f"'{UTIL_PATH}' já está no sys.path.")

print("\nAmbiente configurado e pronto para importar o módulo de utilidades.")

In [ ]:
import experiment_functions as utils
from data import Cleaned as data

In [ ]:
import pandas as pd

CASE_NAME = "case:concept:name"
ACTIVITY_NAME = "concept:name"
TIMESTAMP_NAME = "time:timestamp"

df_1 = pd.read_csv(f"{UTIL_PATH}/data/{data.trabalho1()}.csv.gz")
df_2 = pd.read_csv(f"{UTIL_PATH}/data/{data.trabalho2()}.csv.gz")
df_3 = pd.read_csv(f"{UTIL_PATH}/data/{data.trabalho3()}.csv.gz")
df_1.shape, df_2.shape, df_3.shape

In [ ]:
def format_df_to_pm4py(event_log: pd.DataFrame):
        event_log = event_log.copy()[["processoID", "activity", "dataInicio", "dataFinal"]]
        event_log.rename(columns={
            'processoID': CASE_NAME,
            'activity': ACTIVITY_NAME,
            'dataInicio': 'start_timestamp',
            'dataFinal': TIMESTAMP_NAME
        }, inplace=True)
        event_log[TIMESTAMP_NAME] = pd.to_datetime(event_log[TIMESTAMP_NAME])
        return event_log.sort_values([CASE_NAME, TIMESTAMP_NAME])

df_1['activity'] = df_1['activity'].str.capitalize()
df_1 = format_df_to_pm4py(df_1).reset_index(drop=True)

df_2['activity'] = df_2['activity'].str.capitalize()
df_2 = format_df_to_pm4py(df_2).reset_index(drop=True)

df_3['activity'] = df_3['activity'].str.capitalize()
df_3 = format_df_to_pm4py(df_3).reset_index(drop=True)

In [ ]:
from sklearn.preprocessing import MaxAbsScaler

def get_activity_profile(df: pd.DataFrame):
  activity_profile = df.groupby([CASE_NAME, ACTIVITY_NAME]).size().unstack(fill_value=0)

  scaler = MaxAbsScaler()
  scaler.fit(activity_profile)
  df_scaled = scaler.transform(activity_profile)

  return pd.DataFrame(df_scaled, columns=activity_profile.columns,
                      index=activity_profile.index)

ap_1 = get_activity_profile(df_1)
ap_2 = get_activity_profile(df_2)
ap_3 = get_activity_profile(df_3)

ap_1.shape, ap_2.shape, ap_3.shape

In [ ]:
from sklearn.cluster import DBSCAN, HDBSCAN, AgglomerativeClustering, KMeans
from itertools import product
import numpy as np
import math

def generate_HDBSCAN_search_space(limite, learn_rate, learn_rate_deviation):
    """
    Gera uma sequência de números representando o espaço de busca do parâmetro "min_cluster_size" do HDBSCAN.

    Esta função gera uma sequência de valores que diminui progressivamente, com base em um valor inicial calculado
    a partir de uma fração de `limite`. Os valores são decrementados de acordo com uma taxa de aprendizado
    ajustável (`learn_rate`), que aumenta ao longo do tempo por uma quantidade especificada
    (`learn_rate_deviation`). Isso pode ser utilizado, por exemplo, para definir um intervalo de valores
    de `min_cluster_size` ou `min_samples` na otimização de parâmetros do HDBSCAN.

    Parâmetros:
    -----------
    limite : int
        O limite superior para calcular o valor máximo inicial. O valor inicial é baseado
        em uma fração (5%) desse `limite`, ajustado em 50% para cima.

    learn_rate : float
        A taxa inicial com a qual o valor é reduzido em cada iteração.

    learn_rate_deviation : float
        A quantidade pela qual o `learn_rate` é aumentado após cada iteração, permitindo
        decrementos progressivamente maiores nos valores gerados.

    Retorna:
    --------
    numeros : lista de int
        Uma lista de valores inteiros, cada um representando um valor progressivamente menor,
        gerado ao decrementar o valor máximo de acordo com a dinâmica da taxa de aprendizado.

    """
    numeros = []

    max_val = limite*0.05
    max_val += max_val*0.5
    max_val = math.floor(max_val)

    learn_rate = learn_rate
    val = max_val
    while val > 0:
        numeros.append(val)
        val = math.floor(val - max_val*learn_rate)
        learn_rate += learn_rate_deviation

    return numeros

def get_models_and_configs(activity_profile: pd.DataFrame):
  min_cluster_size_vals = generate_HDBSCAN_search_space(
      len(activity_profile), 0.001, 0.001)

  n_clusters_range_kmeans = range(2, 51, 1)
  n_clusters_range_agglomerative = range(2, 51, 1)

  upper_bound = math.sqrt(len(activity_profile.columns.to_list()))/6
  lower_bound = upper_bound * 0.05
  eps_values = np.arange(upper_bound, lower_bound, -0.01)

  configs = {
      'K-means': {'n_clusters': n_clusters_range_kmeans},
      'DBSCAN': {'eps': eps_values, 'min_samples': [5, 10], "n_jobs": [-1]},
      'AGGLC': {'n_clusters': n_clusters_range_agglomerative, 'linkage': ['ward']},
      'HDBSCAN': {'min_cluster_size': min_cluster_size_vals, "n_jobs": [-1]}
  }

  models = {
      'K-means': KMeans,
      'DBSCAN': DBSCAN,
      'AGGLC': AgglomerativeClustering,
      'HDBSCAN': HDBSCAN
  }

  return models, configs

def get_all_combinations(param_ranges: dict):
  """
  Gets all combinations of parameters from the provided configs dictionary.

  Args:
      configs: A dictionary where keys are arguments for the algorithm and values
      are lists of parameters to try for that argument.

  Returns:
      A list of dictionaries, where each dictionary represents a unique
      combination of parameters for a given algorithm.
  """
  all_combinations = []
  for combination in product(*param_ranges.values()):
    param_dict = dict(zip(param_ranges.keys(), combination))
    all_combinations.append(param_dict)
  return all_combinations

In [ ]:
from sklearn.metrics import silhouette_samples, silhouette_score

def get_valid_silhouette_scores(dataframe, cluster_labels, threshold):
  silhouette_vals = silhouette_samples(dataframe, cluster_labels)

  clusters_info = []
  repr_sil_score = []
  for cluster in filter(lambda x: x != -1, set(cluster_labels)):
    cluster_points = cluster_labels == cluster
    num_points = sum(cluster_points)
    avg_silhouette_cluster = silhouette_vals[cluster_points].mean()

    clusters_info.append({
        'num_points': num_points,
        'avg_silhouette': avg_silhouette_cluster
    })

    if num_points > threshold: # Verifica se o cluster é "representativo"
        repr_sil_score.append(avg_silhouette_cluster)
  return clusters_info, repr_sil_score

def run_model_over_config(model_name, model_factory, config,
                          dataframe, threshold):
    linkage = config.get("linkage")
    max_distance = config.get("eps")
    min_samples = config.get("min_samples")
    min_cluster_size = config.get("min_cluster_size")

    model = model_factory(**config)
    cluster_labels = model.fit_predict(dataframe)

    clusters_info = None
    silhouette_avg = None
    repr_clusters_amount = 0
    avg_repr_silhouette_score = None
    num_clusters = len(list(filter(lambda x: x != -1, set(cluster_labels))))
    if num_clusters > 1:
      silhouette_avg = silhouette_score(dataframe, cluster_labels)
      clusters_info, repr_sil_score = get_valid_silhouette_scores(
          dataframe, cluster_labels, threshold)
      repr_clusters_amount = len(repr_sil_score)
      if repr_clusters_amount > 1:
        avg_repr_silhouette_score = sum(repr_sil_score) / repr_clusters_amount

      return {
          'algo': model_name,
          'linkage': linkage,
          'eps': max_distance,
          'clusters': clusters_info,
          'min_samples': min_samples,
          'num_clusters': num_clusters,
          'silhouette_score': silhouette_avg,
          'min_cluster_size': min_cluster_size,
          'num_representative_clusters': repr_clusters_amount,
          'silhouette_score_representative_clusters': avg_repr_silhouette_score
      }

In [ ]:
from tqdm import tqdm
from math import isnan

def get_cluster_info(row):
  clusters = row['num_clusters']
  repr_clusters = row['num_representative_clusters']
  cluster_info = f"Clusters {repr_clusters}/{clusters}"

  min_cluster_size = row['min_cluster_size']
  if not isnan(min_cluster_size):
    cluster_info += f" | Cluster Size {min_cluster_size}"
  eps = row['eps']
  if not isnan(eps):
    cluster_info += f" | Eps: {round(eps, 2)}"
  linkage = row['linkage']
  if linkage is not None:
    cluster_info += f" | Linkage: {linkage}"
  samples = row['min_samples']
  if not isnan(samples):
    cluster_info += f" | Samples: {samples}"
  return cluster_info

def get_top_params_per_algo(df_name: str, activity_profile: pd.DataFrame):
  models, configs = get_models_and_configs(activity_profile)

  # Colect the results to each parameter
  results = []
  silhouette_thresh = 0.05 * len(activity_profile)

  for model_name, model in models.items():
      config_list = get_all_combinations(configs[model_name])
      print(f"{model_name} with {len(config_list)} combinations.")
      for config in tqdm(config_list, total=len(config_list)):
        model_results = run_model_over_config(model_name, model, config,
                                    activity_profile, silhouette_thresh)
        if model_results: results.append(model_results)

  # Save the results
  results_df = pd.DataFrame(results).sort_values(
      by='silhouette_score', ascending=False
  )
  results_df.to_csv(f"{EXPERIMENT_PATH}/params_search/{df_name}_results.csv", index=False)

  # Select the 5 biggest ones and add a column to params and name
  results_df['params'] = results_df.apply(lambda row: get_cluster_info(row), axis=1)
  top_5_params_per_algo = results_df.groupby('algo').apply(
      lambda group: group.nlargest(5, [
          'silhouette_score_representative_clusters',
          'num_representative_clusters'
      ])
  ).reset_index(drop=True)
  top_5_params_per_algo.rename(columns={
      'silhouette_score_representative_clusters': 'best_silhouette'
  }, inplace=True)
  top_5_params_per_algo["unidade"] = df_name

  return top_5_params_per_algo

In [ ]:
unidade_1 = get_top_params_per_algo("T7U1", ap_1)
unidade_2 = get_top_params_per_algo("T7U2", ap_2)
unidade_3 = get_top_params_per_algo("T7U3", ap_3)

unidade_1.shape, unidade_2.shape, unidade_3.shape

In [ ]:
all_top_5 = pd.concat([unidade_1, unidade_2, unidade_3]).sort_values(
    by=['best_silhouette', 'algo', 'params'], ascending=False
)
all_top_5

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os

unidades = all_top_5["unidade"].unique()

unidades_data = { x: [] for x in unidades }
data_texts = { x: [] for x in unidades }
index_list = []

for (params, algo), data in all_top_5.groupby(["params", "algo"]):
  for unidade in unidades:
    unidade_data = data[data["unidade" ] == unidade]
    if len(unidade_data) > 0:
      unidade_val = unidade_data["best_silhouette"].values[0]
      unidades_data[unidade].append(float(unidade_val))
      data_texts[unidade].append(f"{algo} ({round(unidade_val, 2)})")
    else:
      unidades_data[unidade].append(None)
      data_texts[unidade].append("--")
  index_list.append(params)

table_data = pd.DataFrame(unidades_data, index=index_list, dtype=float)
table_name = pd.DataFrame(data_texts, index=index_list)

clean_data_np = np.array(table_data.values.tolist(), dtype=float)
clean_annot_np = np.array(table_name.values.tolist())

# Garantir que o diretório existe
image_dir = f"{EXPERIMENT_PATH}/params_search/images"
os.makedirs(image_dir, exist_ok=True)
image_path = f"{image_dir}/T7.png"

# Criar a figura explicitamente
fig = plt.figure(figsize=(12, 12))
ax = sns.heatmap(clean_data_np,
    annot=clean_annot_np,
    xticklabels=table_data.columns,
    yticklabels=table_data.index,
    cmap='coolwarm',
    fmt="")

plt.title('Silhouette Score Representativo por Unidade')
plt.ylabel('Parâmetros')

# Salvar usando o objeto fig e garantir que não esteja em branco
plt.savefig(image_path, bbox_inches='tight', dpi=300)
print(f'Imagem salva com sucesso no Drive em: {image_path}')
plt.show()